In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))
df = pd.read_csv(filename)

print("File loaded:", filename)
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

display(df.head())


Saving oil_production_NUPRC_data.csv to oil_production_NUPRC_data.csv
File loaded: oil_production_NUPRC_data.csv
Dataset shape: (417, 7)

Column names:
['production_month', 'terminal_stream', 'liquid_type', 'production_barrels', 'unit', 'source', 'data_status']


,production_month,terminal_stream,liquid_type,production_barrels,unit,source,data_status
0,2025-01-01,BONNY,Crude Oil,7570173,barrels,NUPRC Crude Oil and Condensate Production - 2025,reported
1,2025-02-01,BONNY,Crude Oil,5892185,barrels,NUPRC Crude Oil and Condensate Production - 2025,reported
2,2025-03-01,BONNY,Crude Oil,7360738,barrels,NUPRC Crude Oil and Condensate Production - 2025,reported
3,2025-04-01,BONNY,Crude Oil,7079233,barrels,NUPRC Crude Oil and Condensate Production - 2025,reported
4,2025-05-01,BONNY,Crude Oil,6673050,barrels,NUPRC Crude Oil and Condensate Production - 2025,reported


In [ ]:
# NUPR AI ANALYST - data Quality check

print("EXACT COLUMN NAMES:")
for col in df.columns:
    print(repr(col))

print("\nDATA TYPES AND NON-MISSING VALUES:")
df.info()

print("\nMISSING VALUES:")
display(df.isna().sum().to_frame("Missing Values"))

print("\nDATE RANGE:")
df["production_month"] = pd.to_datetime(
    df["production_month"], errors="coerce"
)
print("Earliest month:", df["production_month"].min())
print("Latest month:", df["production_month"].max())

print("\nLIQUID TYPES:")
display(df["liquid_type"].value_counts(dropna=False))

print("\nTERMINAL STREAMS:")
print("Number of distinct streams:",
      df["terminal_stream"].nunique())
display(df["terminal_stream"].value_counts().head(10))

print("\nPRODUCTION COLUMN CHECK:")
df["Production Barrels"] = pd.to_numeric(
    df["production_barrels"], errors="coerce"
)
print("Total recorded production:",
      f"{df['Production Barrels'].sum():,.0f}")
print("Missing production values:",
      df["Production Barrels"].isna().sum())



EXACT COLUMN NAMES:
'production_month'
'terminal_stream'
'liquid_type'
'production_barrels'
'unit'
'source'
'data_status'

DATA TYPES AND NON-MISSING VALUES:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   production_month    417 non-null    object
 1   terminal_stream     417 non-null    object
 2   liquid_type         417 non-null    object
 3   production_barrels  417 non-null    int64 
 4   unit                417 non-null    object
 5   source              417 non-null    object
 6   data_status         417 non-null    object
dtypes: int64(1), object(6)
memory usage: 22.9+ KB

MISSING VALUES:


,Missing Values
production_month,0
terminal_stream,0
liquid_type,0
production_barrels,0
unit,0
source,0
data_status,0



DATE RANGE:
Earliest month: 2025-01-01 00:00:00
Latest month: 2025-12-01 00:00:00

LIQUID TYPES:


,count
liquid_type,
Crude Oil,320
Condensate,97



TERMINAL STREAMS:
Number of distinct streams: 29


,count
terminal_stream,
BONNY,24
BRASS,24
QUA IBOE,24
ESCRAVOS (Oil Terminal),24
ODUDU (AMENAM BLEND),24
FORCADOS,23
AJAPA,13
OKORO (Ex Ima Terminal),12
TULJA - OKWUIBOME,12



PRODUCTION COLUMN CHECK:
Total recorded production: 595,038,888
Missing production values: 0


In [ ]:
# NUPRC AI ANALYST — CORE ANALYTICS ENGINE

import pandas as pd

# Work on a copy of the cleaned dataset
data = df.copy()

# Normalize text fields
data["terminal_stream"] = data["terminal_stream"].astype(str).str.strip()
data["liquid_type"] = data["liquid_type"].astype(str).str.strip()

# 1. Total production by month
monthly = (
    data.groupby("production_month", as_index=False)
    ["production_barrels"].sum()
    .sort_values("production_month")
)

monthly["MoM Change"] = monthly["production_barrels"].diff()
monthly["MoM Change %"] = (
    monthly["production_barrels"].pct_change() * 100
)

# 2. Total production by terminal
terminal_totals = (
    data.groupby("terminal_stream", as_index=False)
    ["production_barrels"].sum()
    .sort_values("production_barrels", ascending=False)
)

# 3. Terminal production by month
terminal_monthly = (
    data.groupby(
        ["production_month", "terminal_stream"], as_index=False
    )["production_barrels"].sum()
    .sort_values(["terminal_stream", "production_month"])
)

terminal_monthly["MoM Change"] = (
    terminal_monthly.groupby("terminal_stream")["production_barrels"]
    .diff()
)

terminal_monthly["MoM Change %"] = (
    terminal_monthly.groupby("terminal_stream")["production_barrels"]
    .pct_change() * 100
)

# 4. Largest monthly declines across all terminal streams
declines = (
    terminal_monthly.dropna(subset=["MoM Change"])
    .sort_values("MoM Change", ascending=True)
)

print("MONTHLY PRODUCTION AND MONTH-ON-MONTH CHANGES")
display(monthly.round(2))

print("TOP 10 TERMINALS BY TOTAL RECORDED PRODUCTION")
display(terminal_totals.head(10).round(2))

print("10 LARGEST TERMINAL MONTH-ON-MONTH DECLINES")
display(
    declines[
        ["production_month", "terminal_stream",
         "production_barrels", "MoM Change", "MoM Change %"]
    ].head(10).round(2)
)

print("DATASET TOTAL PRODUCTION:",
      f"{data['production_barrels'].sum():,.0f} barrels")


MONTHLY PRODUCTION AND MONTH-ON-MONTH CHANGES


,production_month,production_barrels,MoM Change,MoM Change %
0,2025-01-01,53607462,NaN,NaN
1,2025-02-01,46542078,-7065384.0,-13.18
2,2025-03-01,49436481,2894403.0,6.22
3,2025-04-01,50231801,795320.0,1.61
4,2025-05-01,51096093,864292.0,1.72
5,2025-06-01,50635351,-460742.0,-0.90
6,2025-07-01,52783823,2148472.0,4.24
7,2025-08-01,50276843,-2506980.0,-4.75
8,2025-09-01,46733181,-3543662.0,-7.05
9,2025-10-01,49192821,2459640.0,5.26


TOP 10 TERMINALS BY TOTAL RECORDED PRODUCTION


,terminal_stream,production_barrels
13,FORCADOS,102278027
7,BONNY,88892039
22,QUA IBOE,53026645
12,ESCRAVOS (Oil Terminal),51474661
6,BONGA,44172191
1,AGBAMI,26071647
15,ODUDU (AMENAM BLEND),24765181
11,ERHA,23175712
24,TULJA - OKWUIBOME,22931949
10,EGINA,22859195


10 LARGEST TERMINAL MONTH-ON-MONTH DECLINES


,production_month,terminal_stream,production_barrels,MoM Change,MoM Change %
282,2025-10-01,QUA IBOE,2732790,-1956003.0,-41.72
209,2025-08-01,BONNY,6268468,-1806711.0,-22.37
36,2025-02-01,BONNY,6368067,-1776121.0,-21.81
244,2025-09-01,FORCADOS,7527459,-1471901.0,-16.36
327,2025-12-01,ERHA,348698,-1322088.0,-79.13
129,2025-05-01,FORCADOS,8076640,-1280935.0,-13.69
42,2025-02-01,FORCADOS,7751817,-1110246.0,-12.53
317,2025-12-01,AGBAMI,362617,-1109953.0,-75.38
289,2025-11-01,AGBAMI,1472570,-985594.0,-40.09
71,2025-03-01,FORCADOS,6776256,-975561.0,-12.58


DATASET TOTAL PRODUCTION: 595,038,888 barrels


In [ ]:
from IPython.core import formatters
# DISPLAY THE FIVE LARGEST DECLINES CLLEARLY

cols = ["production_month", "terminal_stream", "production_barrels", "MoM Change", "MoM Change %"]

print(
    declines[cols]
    .head(5)
    .to_string(
        index=False,
        formatters={
            "production_barrels": "{:,.0f}".format, "MoM Change":
            "{:.0f}".format, "MoM Change %":  "{:.2f}%".format
        }
    )
)

production_month terminal_stream production_barrels MoM Change MoM Change %
      2025-10-01        QUA IBOE          2,732,790   -1956003      -41.72%
      2025-08-01           BONNY          6,268,468   -1806711      -22.37%
      2025-02-01           BONNY          6,368,067   -1776121      -21.81%
      2025-09-01        FORCADOS          7,527,459   -1471901      -16.36%
      2025-12-01            ERHA            348,698   -1322088      -79.13%


In [ ]:
# NUPRC AI ANALYST — FIRST QUESTION-ANSWERING ENGINE

def ask_analyst(question):
    q = question.lower().strip()

    # Largest terminal-level decline
    if ("decline" in q or "decrease" in q or "drop" in q):
        result = declines.iloc[0]

        return (
            f"The largest terminal-level month-on-month decline "
            f"in the current ranking is {result['terminal_stream']} "
            f"in {result['production_month']:%B %Y}. "
            f"Production was {result['production_barrels']:,.0f} "
            f"barrels, a change of {result['MoM Change']:,.0f} "
            f"barrels ({result['MoM Change %']:.2f}%)."
        )

    # Total recorded production
    if "total production" in q:
        total = data["production_barrels"].sum()
        return f"Total recorded production in the dataset is {total:,.0f} barrels."

    # Highest-producing terminal
    if "top terminal" in q or "highest-producing terminal" in q:
        top = terminal_totals.iloc[0]
        return (
            f"{top['terminal_stream']} has the highest total recorded "
            f"production: {top['production_barrels']:,.0f} barrels."
        )

    # Latest monthly production
    if "latest month" in q or "latest production" in q:
        latest = monthly.iloc[-1]
        return (
            f"Latest month in the dataset: "
            f"{latest['production_month']:%B %Y}. "
            f"Total production: {latest['production_barrels']:,.0f} barrels."
        )

    return (
        "I don't have a verified answer for that question yet. "
        "Try asking about the largest decline, total production, "
        "the highest-producing terminal, or the latest month."
    )


# Test the analyst
print(ask_analyst("Which terminal had the biggest month-on-month decline?"))
print()
print(ask_analyst("What is the total production?"))
print()
print(ask_analyst("Which is the highest-producing terminal?"))


The largest terminal-level month-on-month decline in the current ranking is QUA IBOE in October 2025. Production was 2,732,790 barrels, a change of -1,956,003 barrels (-41.72%).

Total recorded production in the dataset is 595,038,888 barrels.

FORCADOS has the highest total recorded production: 102,278,027 barrels.


In [ ]:
# NUPRC AI ANALYST — INTERACTIVE CHAT PROTOTYPE

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import pandas as pd

# Automatic overview from the loaded dataset
total_production = data["production_barrels"].sum()
latest_month = monthly.iloc[-1]
top_terminal = terminal_totals.iloc[0]
largest_decline = declines.iloc[0]

overview = f"""
<div style="font-family:Arial,sans-serif;background:#202A2E;color:#FFFFFF;
padding:22px;border-radius:14px;">
  <h2 style="margin-top:0; color: #B88A3B">🇳🇬 NUPRC | AI Production Analyst</h2>
  <p style="color:#B88A3B;">Nigeria Oil Production Intelligence</p>
  <hr style="border-color:#526168;">
  <div style="display:flex;flex-wrap:wrap;gap:20px;">
    <div><small>Total recorded production</small>
    <h2 style="color:#B88A3B;">{total_production:,.0f}</h2><small>barrels · 2025</small></div>
    <div><small>Latest recorded month</small>
    <h2 style="color:#B88A3B;">{latest_month['production_barrels']:,.0f}</h2>
    <small>barrels · {latest_month['production_month']:%B %Y}</small></div>
    <div><small>Top terminal by recorded volume</small>
    <h2 style="color:#B88A3B;">{top_terminal['terminal_stream']}</h2>
    <small>{top_terminal['production_barrels']:,.0f} barrels</small></div>
  </div>
  <hr style="border-color:#526168;">
  <p><b>Key finding:</b> The largest terminal-level decline in the
  current ranking is {largest_decline['terminal_stream']},
  {largest_decline['production_month']:%B %Y}:
  {largest_decline['MoM Change']:,.0f} barrels
  ({largest_decline['MoM Change %']:.2f}%).</p>
  <small>Overview generated from the loaded 2025 CSV. Figures are
  subject to source-data validation.</small>
</div>
"""

display(HTML(overview))

# Chat interface
question_box = widgets.Text(
    placeholder="Ask about production, terminals or declines...",
    description="Question:",
    layout=widgets.Layout(width="80%")
)

ask_button = widgets.Button(
    description="Ask Analyst",
    button_style="primary",
    icon="search"
)

chat_output = widgets.Output()

def handle_question(_):
    question = question_box.value.strip()
    if not question:
        return

    with chat_output:
        print("You:", question)
        print("AI Analyst:", ask_analyst(question))
        print("-" * 65)

    question_box.value = ""

ask_button.on_click(handle_question)

display(widgets.HBox([question_box, ask_button]))
display(chat_output)


Output()

In [ ]:
# NUPRC AI ANALYST — EXPANDED QUESTION ENGINE

def ask_analyst(question):
    q = question.lower().strip()

    # 1. Production by terminal
    if ("by terminal" in q or "by terminal stream" in q) and "total production" in q:
        result = terminal_totals.head(10)
        lines = [
            f"{row['terminal_stream']}: {row['production_barrels']:,.0f} barrels"
            for _, row in result.iterrows()
        ]
        return "Top terminal streams by total recorded production:\n" + "\n".join(lines)

    # 2. Biggest decline
    if any(word in q for word in ["decline", "decrease", "drop", "fall"]):
        result = declines.iloc[0]
        return (
            f"Largest terminal-level month-on-month decline in the current ranking: "
            f"{result['terminal_stream']} in {result['production_month']:%B %Y}.\n"
            f"Production: {result['production_barrels']:,.0f} barrels.\n"
            f"Change: {result['MoM Change']:,.0f} barrels "
            f"({result['MoM Change %']:.2f}%)."
        )

    # 3. Total production
    if "total production" in q or "how much oil" in q:
        total = data["production_barrels"].sum()
        return f"Total recorded production across the loaded dataset: {total:,.0f} barrels."

    # 4. Highest-producing terminal
    if any(word in q for word in ["highest-producing", "highest producing",
                                   "top terminal", "most production"]):
        top = terminal_totals.iloc[0]
        return (
            f"{top['terminal_stream']} has the highest total recorded production: "
            f"{top['production_barrels']:,.0f} barrels."
        )

    # 5. Lowest-producing terminal
    if any(word in q for word in ["lowest-producing", "lowest producing",
                                   "least production"]):
        low = terminal_totals.sort_values("production_barrels").iloc[0]
        return (
            f"{low['terminal_stream']} has the lowest total recorded production: "
            f"{low['production_barrels']:,.0f} barrels."
        )

    # 6. Latest month
    if any(word in q for word in ["latest month", "latest production",
                                   "most recent production"]):
        latest = monthly.iloc[-1]
        return (
            f"Latest month in the dataset: {latest['production_month']:%B %Y}.\n"
            f"Total recorded production: {latest['production_barrels']:,.0f} barrels."
        )

    # 7. Monthly production trend
    if "monthly production" in q or "production by month" in q:
        lines = [
            f"{row['production_month']:%B %Y}: "
            f"{row['production_barrels']:,.0f} barrels"
            for _, row in monthly.iterrows()
        ]
        return "Monthly recorded production:\n" + "\n".join(lines)

    # 8. Crude versus condensate
    if "condensate" in q or "crude" in q:
        liquid_totals = (
            data.groupby("liquid_type")["production_barrels"]
            .sum().sort_values(ascending=False)
        )
        lines = [
            f"{name}: {value:,.0f} barrels"
            for name, value in liquid_totals.items()
        ]
        return "Recorded production by liquid type:\n" + "\n".join(lines)

    # Safe fallback
    return (
        "I don't have a verified answer for that question yet.\n"
        "Try asking about total production, production by terminal, "
        "the biggest decline, monthly production, or crude versus condensate."
    )


print("Updated analyst engine successfully.")
print()
print(ask_analyst("Show total production by terminal stream"))
print()
print(ask_analyst("Compare crude and condensate production"))


Updated analyst engine successfully.

Top terminal streams by total recorded production:
FORCADOS: 102,278,027 barrels
BONNY: 88,892,039 barrels
QUA IBOE: 53,026,645 barrels
ESCRAVOS (Oil Terminal): 51,474,661 barrels
BONGA: 44,172,191 barrels
AGBAMI: 26,071,647 barrels
ODUDU (AMENAM BLEND): 24,765,181 barrels
ERHA: 23,175,712 barrels
TULJA - OKWUIBOME: 22,931,949 barrels
EGINA: 22,859,195 barrels

Recorded production by liquid type:
Crude Oil: 530,007,752 barrels
Condensate: 65,031,136 barrels


In [ ]:
print("Data loaded:", not data.empty)
print("Rows:", len(data))
print("Terminal Streams:", data["terminal_stream"].nunique())
print("Question engine:", callable(ask_analyst))

Data loaded: True
Rows: 417
Terminal Streams: 29
Question engine: True
